# Trendyol 2026 CatBoost GPU Full Run

This is a Kaggle-ready full-run notebook. It rebuilds the text/category signals, trains a higher-iteration CatBoost PU reranker on GPU, predicts all `submission_pairs.csv` rows, and writes an uploadable CSV named `catboost_gpu12000_2026-06-30.csv`.

Do not upload the `_scores.csv` file. The uploadable file has exactly `id,prediction` columns.

In [ ]:
from __future__ import annotations

import csv
import gc
import math
import os
import re
import time
import unicodedata
import warnings
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

try:
    from catboost import CatBoostClassifier, Pool
except ImportError as exc:
    raise RuntimeError(
        "CatBoost is required. On Kaggle, use a GPU notebook image with catboost available "
        "or attach/install catboost before running this notebook."
    ) from exc

try:
    from rapidfuzz import fuzz
except ImportError:
    from difflib import SequenceMatcher

    class _FallbackFuzz:
        @staticmethod
        def ratio(a: str, b: str) -> float:
            return 100.0 * SequenceMatcher(None, a, b).ratio()

        @staticmethod
        def token_set_ratio(a: str, b: str) -> float:
            sa, sb = set(a.split()), set(b.split())
            if not sa or not sb:
                return _FallbackFuzz.ratio(a, b)
            inter = sa & sb
            score = 2.0 * len(inter) / (len(sa) + len(sb))
            return 100.0 * score

        @staticmethod
        def partial_ratio(a: str, b: str) -> float:
            return _FallbackFuzz.ratio(a, b)

    fuzz = _FallbackFuzz()

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    display
except NameError:

    def display(obj):
        print(obj)


SEED = 42
RUN_DATE = "2026-06-30"
METHOD = "catboost_gpu12000"

# Full run defaults. If you need a quick smoke test, set FULL_RUN=False and PREDICT_FULL_SUBMISSION=False.
FULL_RUN = True
PREDICT_FULL_SUBMISSION = True
PRIMARY_POSITIVE_RATE = 0.20
RATE_GRID = [0.18, 0.20, 0.22, 0.24]

CATBOOST_ITERATIONS = 12_000
CATBOOST_LEARNING_RATE = 0.012
CATBOOST_DEPTH = 6
CATBOOST_L2 = 8.0
CATBOOST_RANDOM_STRENGTH = 0.8
CATBOOST_SUBSAMPLE = 0.85
CATBOOST_VERBOSE = 250
USE_GPU = True
CATBOOST_DEVICES = "0"

NEGATIVE_SAMPLE_SCALE = 1.0 if FULL_RUN else 0.12
N_POS = 0 if FULL_RUN else 50_000
LOW_MAX = 2.0
MID_MAX = 4.5
HARD_MAX = 6.5
N_LOW = int(350_000 * NEGATIVE_SAMPLE_SCALE)
N_MID = int(350_000 * NEGATIVE_SAMPLE_SCALE)
N_HARD = int(150_000 * NEGATIVE_SAMPLE_SCALE)

CATEGORY_LEVELS = ("cat_l1", "cat_l2")
CATEGORY_TOPK = 5
CATEGORY_MAX_FEATURES = 70_000
CATEGORY_MAX_ITER = 2_000

PREDICT_CHUNK_SIZE = 150_000 if FULL_RUN else 25_000
PREVIEW_ROW_LIMIT = None if PREDICT_FULL_SUBMISSION else 25_000

REQUIRED_DATA_FILES = {
    "items.csv",
    "terms.csv",
    "training_pairs.csv",
    "submission_pairs.csv",
    "sample_submission.csv",
}
IS_KAGGLE = Path("/kaggle").exists() or bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))


KAGGLE_INPUT_DIR = Path("/kaggle/input")


def has_required_data(path: Path) -> bool:
    return path.is_dir() and all(
        (path / name).is_file() for name in REQUIRED_DATA_FILES
    )


def local_data_dirs():
    cwd = Path.cwd()
    return (cwd / "data", cwd, cwd.parent / "data", cwd.parent)


def candidate_data_dirs():
    yield from local_data_dirs()

    if not (IS_KAGGLE and KAGGLE_INPUT_DIR.is_dir()):
        return

    yield KAGGLE_INPUT_DIR
    yield from (path for path in KAGGLE_INPUT_DIR.iterdir() if path.is_dir())
    yield from (path for path in KAGGLE_INPUT_DIR.glob("*/*") if path.is_dir())
    yield from (path.parent for path in KAGGLE_INPUT_DIR.rglob("items.csv"))


def find_data_dir() -> Path:
    seen: set[Path] = set()
    for candidate in candidate_data_dirs():
        resolved = candidate.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        if has_required_data(resolved):
            return resolved

    searched = ", ".join(str(path) for path in local_data_dirs())
    if IS_KAGGLE:
        searched += ", /kaggle/input/**"
    raise FileNotFoundError(
        f"Could not find the competition CSV files under {searched}"
    )


DATA_DIR = find_data_dir()
ROOT = DATA_DIR.parent if DATA_DIR.name == "data" else DATA_DIR
OUT_DIR = Path("/kaggle/working") if IS_KAGGLE else ROOT / "outputs" / METHOD
OUT_DIR.mkdir(parents=True, exist_ok=True)

LEXICAL_SCORES_PATH = OUT_DIR / f"{METHOD}_{RUN_DATE}_lexical_scores.csv"
TERM_CATEGORY_TOPK_PATH = OUT_DIR / f"{METHOD}_{RUN_DATE}_term_category_topk.csv"
MODEL_PATH = OUT_DIR / f"{METHOD}_{RUN_DATE}.cbm"
SCORE_PATH = OUT_DIR / f"{METHOD}_{RUN_DATE}_scores.csv"
PRIMARY_SUBMISSION_PATH = OUT_DIR / f"{METHOD}_{RUN_DATE}.csv"

print(f"data_dir={DATA_DIR}")
print(f"out_dir={OUT_DIR}")
print(f"full_run={FULL_RUN}")
print(f"catboost_iterations={CATBOOST_ITERATIONS:,}")
print(f"primary_submission={PRIMARY_SUBMISSION_PATH}")

## Load Data

The training file is positive-only. The model samples likely negatives from the unlabeled candidate pool using lexical-score bands, then trains a PU-style classifier.

In [ ]:
def count_csv_rows(path: Path) -> int:
    with path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.reader(handle)
        next(reader, None)
        return sum(1 for _ in reader)


files = [DATA_DIR / name for name in sorted(REQUIRED_DATA_FILES)]
audit = pd.DataFrame(
    {
        "file": [path.name for path in files],
        "rows": [count_csv_rows(path) for path in files],
        "size_mb": [round(path.stat().st_size / 1024 / 1024, 2) for path in files],
    }
)
display(audit)

terms_df = pd.read_csv(DATA_DIR / "terms.csv")
items_meta = pd.read_csv(
    DATA_DIR / "items.csv", usecols=["item_id", "category", "gender", "age_group"]
)
train_pairs_all = pd.read_csv(
    DATA_DIR / "training_pairs.csv", usecols=["term_id", "item_id"]
)

print(f"terms={len(terms_df):,}")
print(f"items={len(items_meta):,}")
print(f"positive_pairs={len(train_pairs_all):,}")

## Lexical And Starter-Overlap Features

These features combine the previous lexical scorer with the starter notebook's Turkish token overlap. The overlap features are useful because the public `.79` analysis showed that good submissions still need to keep many simple query/product token matches.

In [ ]:
TOKEN_RE = re.compile(r"[a-z0-9]+")
TR_TOKEN_SPLIT_RE = re.compile(r"[^0-9a-zçğıöşü]+")

ASCII_TRANSLATION = str.maketrans(
    {
        "ç": "c",
        "ğ": "g",
        "ı": "i",
        "ö": "o",
        "ş": "s",
        "ü": "u",
        "â": "a",
        "î": "i",
        "û": "u",
        "Ç": "c",
        "Ğ": "g",
        "İ": "i",
        "I": "i",
        "Ö": "o",
        "Ş": "s",
        "Ü": "u",
        "Â": "a",
        "Î": "i",
        "Û": "u",
    }
)

STOPWORDS = {
    "a",
    "ve",
    "ile",
    "icin",
    "bir",
    "adet",
    "ad",
    "li",
    "set",
    "seti",
    "takim",
    "takimi",
    "urun",
    "model",
    "uyumlu",
    "orjinal",
    "orijinal",
}

COLORS = {
    "beyaz",
    "siyah",
    "kirmizi",
    "mavi",
    "yesil",
    "sari",
    "pembe",
    "mor",
    "gri",
    "lacivert",
    "kahverengi",
    "bej",
    "krem",
    "turuncu",
    "bordo",
    "gold",
    "altin",
    "gumus",
    "haki",
    "taba",
    "lila",
    "ekru",
    "vizon",
    "fume",
}

GENDER_ALLOWED = {
    "kadin": {"kadin", "unisex", "unknown"},
    "bayan": {"kadin", "unisex", "unknown"},
    "erkek": {"erkek", "unisex", "unknown"},
    "unisex": {"kadin", "erkek", "unisex", "unknown"},
}

AGE_ALLOWED = {
    "bebek": {"bebek", "unknown"},
    "cocuk": {"cocuk", "bebek", "unknown"},
    "kiz": {"cocuk", "bebek", "unknown"},
}

BASE_FEATURE_NAMES = [
    "n_query_tokens",
    "title_cov",
    "category_cov",
    "brand_cov",
    "attr_cov",
    "full_cov",
    "title_ratio",
    "category_ratio",
    "brand_ratio",
    "query_in_title",
    "query_in_category",
    "query_in_brand",
    "full_complete",
    "full_high",
    "title_complete",
    "category_complete",
    "brand_complete",
    "short_complete",
    "gender_mismatch",
    "age_mismatch",
    "color_mismatch",
    "weak_match",
    "title_len",
    "category_len",
    "brand_len",
    "attr_len",
    "starter_overlap",
    "starter_overlap_ratio",
    "starter_overlap_ge1",
    "starter_overlap_ge2",
    "starter_query_len",
    "lexical_score",
]


def normalize_text(value: object) -> str:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ""
    text = str(value).translate(ASCII_TRANSLATION).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.replace("&", " ")
    return re.sub(r"\s+", " ", text).strip()


def normalize_tr(value: object) -> str:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ""
    text = str(value).replace("İ", "i").replace("I", "ı")
    return text.lower().replace("i̇", "i")


def starter_tokenize(value: object, min_len: int = 2) -> set[str]:
    return {
        token
        for token in TR_TOKEN_SPLIT_RE.split(normalize_tr(value))
        if len(token) >= min_len
    }


def stem_token(token: str) -> str:
    if len(token) <= 4:
        return token
    for suffix in (
        "larin",
        "lerin",
        "lari",
        "leri",
        "lara",
        "lere",
        "larda",
        "lerde",
        "ndan",
        "nden",
        "dan",
        "den",
        "nin",
        "nun",
        "ler",
        "lar",
        "lik",
        "luk",
        "cuk",
        "cik",
        "ci",
        "cu",
        "si",
        "su",
        "u",
        "i",
    ):
        if token.endswith(suffix) and len(token) - len(suffix) >= 3:
            return token[: -len(suffix)]
    return token


@lru_cache(maxsize=1_000_000)
def token_set(text: str) -> frozenset[str]:
    out: set[str] = set()
    for token in TOKEN_RE.findall(text):
        if token in STOPWORDS:
            continue
        out.add(token)
        stemmed = stem_token(token)
        if stemmed not in STOPWORDS:
            out.add(stemmed)
    return frozenset(out)


def query_tokens(text: str) -> tuple[str, ...]:
    seen: set[str] = set()
    out: list[str] = []
    for token in TOKEN_RE.findall(text):
        if token in STOPWORDS:
            continue
        stemmed = stem_token(token)
        for candidate in (token, stemmed):
            if candidate and candidate not in STOPWORDS and candidate not in seen:
                seen.add(candidate)
                out.append(candidate)
                break
    return tuple(out)


def coverage(query: tuple[str, ...], field: frozenset[str]) -> float:
    if not query:
        return 0.0
    return sum(1 for token in query if token in field) / len(query)


def color_tokens(query: tuple[str, ...]) -> set[str]:
    return {token for token in query if token in COLORS}


def load_terms(path: Path) -> dict[str, dict[str, object]]:
    terms = pd.read_csv(path)
    records: dict[str, dict[str, object]] = {}
    for row in terms.itertuples(index=False):
        raw_query = "" if pd.isna(row.query) else str(row.query)
        norm_query = normalize_text(raw_query)
        q_tokens = query_tokens(norm_query)
        records[row.term_id] = {
            "query_raw": raw_query,
            "query": norm_query,
            "tokens": q_tokens,
            "colors": color_tokens(q_tokens),
            "starter_tokens": starter_tokenize(raw_query),
        }
    return records


def load_items(path: Path) -> dict[str, tuple[str, str, str, str, str, str, str]]:
    items: dict[str, tuple[str, str, str, str, str, str, str]] = {}
    with path.open(newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        for row in tqdm(reader, desc="items"):
            item_id = row["item_id"]
            raw_parts = [
                row.get("title") or "",
                row.get("category") or "",
                row.get("brand") or "",
                row.get("gender") or "",
                row.get("age_group") or "",
                row.get("attributes") or "",
            ]
            items[item_id] = (
                normalize_text(row.get("title")),
                normalize_text(row.get("category")),
                normalize_text(row.get("brand")),
                normalize_text(row.get("gender")) or "unknown",
                normalize_text(row.get("age_group")) or "unknown",
                normalize_text(row.get("attributes")),
                " ".join(raw_parts),
            )
    return items


def lexical_score_from_features(features: dict[str, float]) -> float:
    score = 0.0
    score += 2.1 * features["full_cov"]
    score += 1.4 * features["title_cov"]
    score += 1.0 * features["category_cov"]
    score += 1.2 * features["brand_cov"]
    score += 0.55 * features["attr_cov"]
    score += 0.75 * features["title_ratio"]
    score += 0.45 * features["category_ratio"]
    score += 0.55 * features["brand_ratio"]
    if features["query_in_title"]:
        score += 1.0
    if features["query_in_category"]:
        score += 0.8
    if features["query_in_brand"]:
        score += 1.1
    if features["full_complete"]:
        score += 1.2
    elif features["full_high"]:
        score += 0.55
    if features["title_complete"]:
        score += 0.9
    if features["category_complete"]:
        score += 0.7
    if features["brand_complete"]:
        score += 0.9
    if features["short_complete"]:
        score += 0.55
    if features["gender_mismatch"]:
        score -= 0.85
    if features["age_mismatch"]:
        score -= 0.75
    if features["color_mismatch"]:
        score -= 0.7
    if features["weak_match"]:
        score -= 0.9
    return score


def pair_features(
    term: dict[str, object], item: tuple[str, str, str, str, str, str, str]
) -> dict[str, float]:
    query = term["query"]
    q_tokens = term["tokens"]
    q_colors = term["colors"]
    q_starter = term["starter_tokens"]
    title, category, brand, gender, age_group, attrs, starter_text = item

    title_tokens = token_set(title)
    category_tokens = token_set(category)
    brand_tokens = token_set(brand)
    attr_tokens = token_set(attrs)
    full_tokens = title_tokens | category_tokens | brand_tokens | attr_tokens

    title_cov = coverage(q_tokens, title_tokens)
    category_cov = coverage(q_tokens, category_tokens)
    brand_cov = coverage(q_tokens, brand_tokens)
    attr_cov = coverage(q_tokens, attr_tokens)
    full_cov = coverage(q_tokens, full_tokens)

    title_ratio = fuzz.token_set_ratio(query, title) / 100.0
    category_ratio = fuzz.token_set_ratio(query, category) / 100.0
    brand_ratio = max(
        fuzz.ratio(query, brand) / 100.0,
        fuzz.partial_ratio(query, brand) / 100.0 if brand else 0.0,
    )

    query_in_title = float(bool(query and query in title))
    query_in_category = float(bool(query and query in category))
    query_in_brand = float(bool(query and query in brand))
    full_complete = float(full_cov >= 0.999)
    full_high = float(0.75 <= full_cov < 0.999)
    title_complete = float(title_cov >= 0.999)
    category_complete = float(category_cov >= 0.999)
    brand_complete = float(brand_cov >= 0.999 and bool(brand))
    short_complete = float(len(q_tokens) <= 2 and full_cov >= 0.999)

    gender_mismatch = 0.0
    for gender_word, allowed in GENDER_ALLOWED.items():
        if (
            gender_word in q_tokens
            and gender not in allowed
            and gender_word not in full_tokens
        ):
            gender_mismatch = 1.0
            break

    age_mismatch = 0.0
    for age_word, allowed in AGE_ALLOWED.items():
        if (
            age_word in q_tokens
            and age_group not in allowed
            and age_word not in full_tokens
        ):
            age_mismatch = 1.0
            break

    color_mismatch = float(bool(q_colors and full_tokens.isdisjoint(q_colors)))
    weak_match = float(
        full_cov < 0.34 and max(title_ratio, category_ratio, brand_ratio) < 0.55
    )

    item_starter_tokens = starter_tokenize(starter_text)
    starter_overlap = len(q_starter & item_starter_tokens) if q_starter else 0
    starter_query_len = len(q_starter)
    starter_overlap_ratio = (
        starter_overlap / starter_query_len if starter_query_len else 0.0
    )

    features = {
        "n_query_tokens": float(len(q_tokens)),
        "title_cov": title_cov,
        "category_cov": category_cov,
        "brand_cov": brand_cov,
        "attr_cov": attr_cov,
        "full_cov": full_cov,
        "title_ratio": title_ratio,
        "category_ratio": category_ratio,
        "brand_ratio": brand_ratio,
        "query_in_title": query_in_title,
        "query_in_category": query_in_category,
        "query_in_brand": query_in_brand,
        "full_complete": full_complete,
        "full_high": full_high,
        "title_complete": title_complete,
        "category_complete": category_complete,
        "brand_complete": brand_complete,
        "short_complete": short_complete,
        "gender_mismatch": gender_mismatch,
        "age_mismatch": age_mismatch,
        "color_mismatch": color_mismatch,
        "weak_match": weak_match,
        "title_len": float(len(title_tokens)),
        "category_len": float(len(category_tokens)),
        "brand_len": float(len(brand_tokens)),
        "attr_len": float(len(attr_tokens)),
        "starter_overlap": float(starter_overlap),
        "starter_overlap_ratio": float(starter_overlap_ratio),
        "starter_overlap_ge1": float(starter_overlap >= 1),
        "starter_overlap_ge2": float(starter_overlap >= 2),
        "starter_query_len": float(starter_query_len),
    }
    features["lexical_score"] = lexical_score_from_features(features)
    return features


print(f"base feature count={len(BASE_FEATURE_NAMES)}")

## Query-To-Category Signal

This reproduces the category-prior signal: train a small char-ngram classifier from positive training pairs, predict top categories for every term, and use the candidate item's category rank/probability as model features.

In [ ]:
def split_category(value: object) -> tuple[str, str, str, str]:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        parts = ["unknown"]
    else:
        parts = [normalize_text(part) for part in str(value).split("/")]
        parts = [part for part in parts if part]
        if not parts:
            parts = ["unknown"]

    cat_l1 = parts[0]
    cat_l2 = "/".join(parts[:2]) if len(parts) >= 2 else cat_l1
    cat_l3 = "/".join(parts[:3]) if len(parts) >= 3 else cat_l2
    cat_leaf = "/".join(parts)
    return cat_l1, cat_l2, cat_l3, cat_leaf


def add_category_levels(
    frame: pd.DataFrame, category_col: str = "category"
) -> pd.DataFrame:
    levels = pd.DataFrame(
        frame[category_col].map(split_category).tolist(),
        columns=["cat_l1", "cat_l2", "cat_l3", "cat_leaf"],
        index=frame.index,
    )
    return pd.concat([frame, levels], axis=1)


def build_category_examples(
    terms: pd.DataFrame,
    items: pd.DataFrame,
    train_pairs: pd.DataFrame,
    max_examples: int | None = None,
    seed: int = SEED,
) -> pd.DataFrame:
    examples = (
        train_pairs.merge(terms, on="term_id", how="left")
        .merge(items[["item_id", "category"]], on="item_id", how="left")
        .dropna(subset=["query", "category"])
    )
    examples = add_category_levels(examples)
    examples["query_norm"] = examples["query"].map(normalize_text)
    examples = examples.drop_duplicates(["query_norm", "cat_l2"])
    if max_examples is not None and len(examples) > max_examples:
        examples = examples.sample(n=max_examples, random_state=seed)
    return examples[
        ["query_norm", "cat_l1", "cat_l2", "cat_l3", "cat_leaf"]
    ].reset_index(drop=True)


class CategorySignal:
    def __init__(
        self,
        levels: tuple[str, ...] = CATEGORY_LEVELS,
        topk: int = CATEGORY_TOPK,
        max_features: int = CATEGORY_MAX_FEATURES,
        max_iter: int = CATEGORY_MAX_ITER,
        min_class_count: int = 2,
        seed: int = SEED,
    ):
        self.levels = levels
        self.topk = topk
        self.max_features = max_features
        self.max_iter = max_iter
        self.min_class_count = min_class_count
        self.seed = seed
        self.vectorizer: TfidfVectorizer | None = None
        self.models: dict[str, SGDClassifier] = {}

    def fit(self, examples: pd.DataFrame) -> "CategorySignal":
        self.vectorizer = TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(2, 5),
            min_df=2,
            max_features=self.max_features,
            dtype=np.float32,
            sublinear_tf=True,
        )
        x_all = self.vectorizer.fit_transform(examples["query_norm"])
        print(f"category tfidf shape={x_all.shape}")

        for level in self.levels:
            counts = examples[level].value_counts()
            keep_labels = counts.index[counts >= self.min_class_count]
            keep = examples[level].isin(keep_labels).to_numpy()
            y = examples.loc[keep, level].to_numpy()
            x = x_all[keep]
            clf = SGDClassifier(
                loss="log_loss",
                penalty="l2",
                alpha=2e-5,
                max_iter=self.max_iter,
                tol=1e-3,
                class_weight="balanced",
                random_state=self.seed,
                n_jobs=-1,
            )
            start = time.time()
            clf.fit(x, y)
            self.models[level] = clf
            print(
                f"{level}: classes={len(clf.classes_):,} examples={len(y):,} seconds={time.time() - start:.1f}"
            )
        return self

    def predict_terms(
        self, terms: pd.DataFrame, chunk_size: int = 25_000
    ) -> pd.DataFrame:
        if self.vectorizer is None:
            raise RuntimeError("fit the category signal first")

        term_ids = terms["term_id"].to_numpy()
        queries = terms["query"].map(normalize_text).to_numpy()
        chunks: list[pd.DataFrame] = []

        for start in tqdm(range(0, len(terms), chunk_size), desc="category topk"):
            end = min(start + chunk_size, len(terms))
            x = self.vectorizer.transform(queries[start:end])
            out: dict[str, object] = {"term_id": term_ids[start:end]}

            for level, clf in self.models.items():
                proba = clf.predict_proba(x)
                k = min(self.topk, proba.shape[1])
                idx = np.argpartition(proba, -k, axis=1)[:, -k:]
                local_order = np.argsort(
                    -np.take_along_axis(proba, idx, axis=1), axis=1
                )
                idx = np.take_along_axis(idx, local_order, axis=1)
                scores = np.take_along_axis(proba, idx, axis=1).astype(np.float32)
                labels = clf.classes_[idx]
                for rank in range(k):
                    out[f"{level}_top{rank + 1}"] = labels[:, rank]
                    out[f"{level}_prob{rank + 1}"] = scores[:, rank]

            chunks.append(pd.DataFrame(out))

        return pd.concat(chunks, ignore_index=True)


category_examples = build_category_examples(terms_df, items_meta, train_pairs_all)
print(category_examples.shape)
for col in ["cat_l1", "cat_l2", "cat_l3", "cat_leaf"]:
    print(f"{col}: {category_examples[col].nunique():,}")

category_signal = CategorySignal().fit(category_examples)
term_category_topk = category_signal.predict_terms(terms_df)
term_category_topk.to_csv(TERM_CATEGORY_TOPK_PATH, index=False)
print(f"wrote {TERM_CATEGORY_TOPK_PATH}")
display(term_category_topk.head())

## Build CatBoost Feature Frames

The matrix includes lexical features, starter-overlap features, category-prior features, and item metadata categoricals. Raw `term_id` is intentionally not used because train and submission terms do not overlap.

In [ ]:
CATEGORY_FEATURE_NAMES: list[str] = []
for level in CATEGORY_LEVELS:
    CATEGORY_FEATURE_NAMES.extend(
        [
            f"{level}_top1_match",
            f"{level}_top3_match",
            f"{level}_top5_match",
            f"{level}_candidate_prob",
            f"{level}_top1_prob",
            f"{level}_rank",
        ]
    )

ITEM_CAT_FEATURES = ["item_cat_l1", "item_cat_l2", "item_gender", "item_age_group"]
FEATURE_NAMES = BASE_FEATURE_NAMES + CATEGORY_FEATURE_NAMES + ITEM_CAT_FEATURES
CAT_FEATURES = ITEM_CAT_FEATURES


def build_term_category_lookup(
    topk: pd.DataFrame,
    levels: tuple[str, ...] = CATEGORY_LEVELS,
    k: int = CATEGORY_TOPK,
) -> dict[str, dict[str, object]]:
    lookup: dict[str, dict[str, object]] = {}
    for row in topk.itertuples(index=False):
        row_dict = row._asdict()
        entry: dict[str, object] = {}
        for level in levels:
            scores: dict[str, float] = {}
            for rank in range(1, k + 1):
                label = row_dict.get(f"{level}_top{rank}")
                score = row_dict.get(f"{level}_prob{rank}")
                if isinstance(label, str) and label:
                    scores[label] = float(score)
            entry[level] = scores
            entry[f"{level}_top1_prob"] = next(iter(scores.values()), 0.0)
        lookup[row_dict["term_id"]] = entry
    return lookup


def build_item_meta_lookup(items: pd.DataFrame) -> dict[str, dict[str, str]]:
    items = add_category_levels(items.copy())
    lookup: dict[str, dict[str, str]] = {}
    for row in tqdm(
        items.itertuples(index=False), total=len(items), desc="item metadata"
    ):
        lookup[row.item_id] = {
            "cat_l1": row.cat_l1,
            "cat_l2": row.cat_l2,
            "item_cat_l1": row.cat_l1 or "unknown",
            "item_cat_l2": row.cat_l2 or "unknown",
            "item_gender": normalize_text(row.gender) or "unknown",
            "item_age_group": normalize_text(row.age_group) or "unknown",
        }
    return lookup


def category_pair_features(
    term_id: str,
    item_id: str,
    term_category_lookup: dict[str, dict[str, object]],
    item_meta_lookup: dict[str, dict[str, str]],
) -> dict[str, float | str]:
    term_entry = term_category_lookup.get(term_id, {})
    item_entry = item_meta_lookup.get(item_id, {})
    out: dict[str, float | str] = {
        "item_cat_l1": item_entry.get("item_cat_l1", "unknown"),
        "item_cat_l2": item_entry.get("item_cat_l2", "unknown"),
        "item_gender": item_entry.get("item_gender", "unknown"),
        "item_age_group": item_entry.get("item_age_group", "unknown"),
    }

    for level in CATEGORY_LEVELS:
        scores = term_entry.get(level, {})
        if not isinstance(scores, dict):
            scores = {}
        ordered_labels = list(scores.keys())
        candidate = item_entry.get(level, "")
        rank = ordered_labels.index(candidate) + 1 if candidate in scores else 0

        out[f"{level}_top1_match"] = float(rank == 1)
        out[f"{level}_top3_match"] = float(1 <= rank <= 3)
        out[f"{level}_top5_match"] = float(1 <= rank <= 5)
        out[f"{level}_candidate_prob"] = float(scores.get(candidate, 0.0))
        out[f"{level}_top1_prob"] = float(term_entry.get(f"{level}_top1_prob", 0.0))
        out[f"{level}_rank"] = float(rank)

    return out


def rows_to_frame(
    rows: pd.DataFrame,
    terms: dict[str, dict[str, object]],
    items: dict[str, tuple[str, str, str, str, str, str, str]],
    term_category_lookup: dict[str, dict[str, object]],
    item_meta_lookup: dict[str, dict[str, str]],
    desc: str,
) -> pd.DataFrame:
    numeric_names = BASE_FEATURE_NAMES + CATEGORY_FEATURE_NAMES
    numeric = np.empty((len(rows), len(numeric_names)), dtype=np.float32)
    cat_values = {name: np.empty(len(rows), dtype=object) for name in ITEM_CAT_FEATURES}

    iterator = tqdm(
        rows.itertuples(index=False), total=len(rows), desc=desc, leave=False
    )
    for i, row in enumerate(iterator):
        features: dict[str, float | str] = pair_features(
            terms[row.term_id], items[row.item_id]
        )
        features.update(
            category_pair_features(
                row.term_id, row.item_id, term_category_lookup, item_meta_lookup
            )
        )
        numeric[i] = [float(features.get(name, 0.0)) for name in numeric_names]
        for name in ITEM_CAT_FEATURES:
            cat_values[name][i] = features.get(name, "unknown") or "unknown"

    frame = pd.DataFrame(numeric, columns=numeric_names)
    for name in ITEM_CAT_FEATURES:
        frame[name] = cat_values[name].astype(str)
    return frame[FEATURE_NAMES]


term_category_lookup = build_term_category_lookup(term_category_topk)
item_meta_lookup = build_item_meta_lookup(items_meta)
print(
    f"feature count={len(FEATURE_NAMES)} base={len(BASE_FEATURE_NAMES)} category={len(CATEGORY_FEATURE_NAMES)} cat={len(CAT_FEATURES)}"
)

## Negative Sampling

The candidate pool is scored lexically, then sampled from easy, medium, and hard bands. This matches the local PU setup while keeping all known positives from `training_pairs.csv`.

In [ ]:
def sample_frame(
    frame: pd.DataFrame, mask: pd.Series, n: int, seed: int
) -> pd.DataFrame:
    subset = frame.loc[mask]
    if len(subset) <= n:
        return subset
    return subset.sample(n=n, random_state=seed)


def write_lexical_scores(
    output_path: Path,
    terms: dict[str, dict[str, object]],
    items: dict[str, tuple[str, str, str, str, str, str, str]],
    chunk_size: int = 150_000,
) -> Path:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    written = 0
    with output_path.open("w", newline="", encoding="utf-8") as out_handle:
        writer = csv.writer(out_handle)
        writer.writerow(["id", "term_id", "score"])
        reader = pd.read_csv(DATA_DIR / "submission_pairs.csv", chunksize=chunk_size)
        for chunk in tqdm(reader, desc="lexical scores"):
            for row in chunk.itertuples(index=False):
                score = pair_features(terms[row.term_id], items[row.item_id])[
                    "lexical_score"
                ]
                writer.writerow([row.id, row.term_id, f"{score:.6f}"])
            written += len(chunk)
            del chunk
            gc.collect()
    print(f"wrote {output_path} rows={written:,}")
    return output_path


def build_unlabeled_negatives(
    lexical_scores_path: Path, seed: int = SEED
) -> pd.DataFrame:
    pairs = pd.read_csv(
        DATA_DIR / "submission_pairs.csv", usecols=["id", "term_id", "item_id"]
    )
    scores = pd.read_csv(lexical_scores_path, usecols=["score"])
    if len(pairs) != len(scores):
        raise ValueError("submission_pairs and lexical_scores row counts differ")

    pairs["score"] = scores["score"].astype("float32").to_numpy()
    parts = [
        sample_frame(pairs, pairs["score"] < LOW_MAX, N_LOW, seed),
        sample_frame(
            pairs,
            (pairs["score"] >= LOW_MAX) & (pairs["score"] < MID_MAX),
            N_MID,
            seed + 1,
        ),
        sample_frame(
            pairs,
            (pairs["score"] >= MID_MAX) & (pairs["score"] < HARD_MAX),
            N_HARD,
            seed + 2,
        ),
    ]
    negatives = pd.concat(parts, ignore_index=True).drop_duplicates("id")
    print("negative score distribution")
    print(negatives["score"].describe(percentiles=[0.1, 0.5, 0.9, 0.99]).to_string())
    return negatives[["term_id", "item_id"]]


terms = load_terms(DATA_DIR / "terms.csv")
items = load_items(DATA_DIR / "items.csv")

if LEXICAL_SCORES_PATH.exists():
    print(f"using existing lexical scores: {LEXICAL_SCORES_PATH}")
else:
    write_lexical_scores(LEXICAL_SCORES_PATH, terms, items)

positives = train_pairs_all.copy()
if N_POS and len(positives) > N_POS:
    positives = positives.sample(n=N_POS, random_state=SEED)
negatives = build_unlabeled_negatives(LEXICAL_SCORES_PATH)

x_pos = rows_to_frame(
    positives, terms, items, term_category_lookup, item_meta_lookup, "positive features"
)
x_neg = rows_to_frame(
    negatives, terms, items, term_category_lookup, item_meta_lookup, "negative features"
)
x = pd.concat([x_pos, x_neg], ignore_index=True)
y = np.concatenate(
    [np.ones(len(x_pos), dtype=np.int8), np.zeros(len(x_neg), dtype=np.int8)]
)

print(f"matrix={x.shape} positives={int(y.sum()):,} positive_rate={y.mean():.4f}")
del x_pos, x_neg
_ = gc.collect()

## Train CatBoost On GPU

This is the high-iteration run: `12000` iterations, lower learning rate, category metadata as CatBoost categoricals, and no raw `term_id` leakage.

In [ ]:
x_train, x_valid, y_train, y_valid = train_test_split(
    x,
    y,
    test_size=0.15,
    random_state=SEED,
    stratify=y,
)
del x
_ = gc.collect()

params = {
    "loss_function": "Logloss",
    "eval_metric": "Logloss",
    "iterations": CATBOOST_ITERATIONS,
    "learning_rate": CATBOOST_LEARNING_RATE,
    "depth": CATBOOST_DEPTH,
    "l2_leaf_reg": CATBOOST_L2,
    "random_strength": CATBOOST_RANDOM_STRENGTH,
    "bootstrap_type": "Bernoulli",
    "subsample": CATBOOST_SUBSAMPLE,
    "auto_class_weights": "Balanced",
    "random_seed": SEED,
    "allow_writing_files": False,
    "verbose": CATBOOST_VERBOSE,
}
if USE_GPU:
    params.update(
        {"task_type": "GPU", "devices": CATBOOST_DEVICES, "gpu_ram_part": 0.90}
    )
else:
    params.update({"thread_count": -1})

model = CatBoostClassifier(**params)
train_pool = Pool(x_train, y_train, cat_features=CAT_FEATURES)
valid_pool = Pool(x_valid, y_valid, cat_features=CAT_FEATURES)

start = time.time()
model.fit(train_pool, eval_set=valid_pool, use_best_model=False)
print(f"training_minutes={(time.time() - start) / 60:.1f}")

valid_prob = model.predict_proba(valid_pool)[:, 1]
best = (0.0, 0.5)
for threshold in np.linspace(0.05, 0.95, 91):
    pred = (valid_prob >= threshold).astype(np.int8)
    score = f1_score(y_valid, pred, average="macro")
    if score > best[0]:
        best = (float(score), float(threshold))
print(f"synthetic valid macro_f1={best[0]:.5f} threshold={best[1]:.3f}")

importances = pd.Series(
    model.get_feature_importance(valid_pool), index=FEATURE_NAMES
).sort_values(ascending=False)
display(importances.head(40).to_frame("importance"))

model.save_model(str(MODEL_PATH))
print(f"wrote {MODEL_PATH}")

## Predict And Write Uploadable CSV

The `_scores.csv` file is for ranking/debug only and is not uploadable. The primary upload file is `catboost_gpu12000_2026-06-30.csv` with exactly `id,prediction` columns.

In [ ]:
def predict_pairs_to_file(
    model: CatBoostClassifier,
    output_path: Path,
    chunk_size: int = PREDICT_CHUNK_SIZE,
    row_limit: int | None = PREVIEW_ROW_LIMIT,
) -> Path:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    written = 0
    with output_path.open("w", newline="", encoding="utf-8") as out_handle:
        writer = csv.writer(out_handle)
        writer.writerow(["id", "term_id", "prob"])

        reader = pd.read_csv(DATA_DIR / "submission_pairs.csv", chunksize=chunk_size)
        for chunk in tqdm(reader, desc="predict chunks"):
            if row_limit is not None:
                remaining = row_limit - written
                if remaining <= 0:
                    break
                chunk = chunk.head(remaining)

            matrix = rows_to_frame(
                chunk,
                terms,
                items,
                term_category_lookup,
                item_meta_lookup,
                "chunk features",
            )
            pool = Pool(matrix, cat_features=CAT_FEATURES)
            prob = model.predict_proba(pool)[:, 1]
            writer.writerows(
                (row_id, term_id, f"{score:.8f}")
                for row_id, term_id, score in zip(chunk["id"], chunk["term_id"], prob)
            )
            written += len(chunk)
            del matrix, pool, prob, chunk
            gc.collect()

    print(f"wrote {output_path} rows={written:,}")
    return output_path


def make_rate_submission(
    scores_path: Path, output_path: Path, rate: float = PRIMARY_POSITIVE_RATE
) -> Path:
    scores = pd.read_csv(scores_path, usecols=["id", "prob"])
    n_pos = int(round(len(scores) * rate))
    order = np.argsort(-scores["prob"].to_numpy(), kind="mergesort")
    pred = np.zeros(len(scores), dtype=np.int8)
    pred[order[:n_pos]] = 1
    submission = pd.DataFrame({"id": scores["id"].to_numpy(), "prediction": pred})
    submission.to_csv(output_path, index=False)
    print(
        f"{output_path.name}: positives={int(pred.sum()):,} rate={pred.mean():.5f} rows={len(pred):,}"
    )
    return output_path


predict_pairs_to_file(model, SCORE_PATH)

if PREDICT_FULL_SUBMISSION:
    for rate in RATE_GRID:
        suffix = (
            ""
            if abs(rate - PRIMARY_POSITIVE_RATE) < 1e-12
            else f"_r{int(rate * 100):02d}"
        )
        out_path = OUT_DIR / f"{METHOD}_{RUN_DATE}{suffix}.csv"
        make_rate_submission(SCORE_PATH, out_path, rate=rate)

    sub = pd.read_csv(PRIMARY_SUBMISSION_PATH)
    expected = pd.read_csv(DATA_DIR / "sample_submission.csv", nrows=5)
    assert list(sub.columns) == ["id", "prediction"], sub.columns.tolist()
    assert len(sub) == count_csv_rows(DATA_DIR / "submission_pairs.csv")
    assert set(sub["prediction"].unique()).issubset({0, 1})
    print("primary upload file:", PRIMARY_SUBMISSION_PATH)
    print("do not upload scores file:", SCORE_PATH)
    display(sub.head())
else:
    display(pd.read_csv(SCORE_PATH).head())